# Extending the STEM Digital Twin — abTEM, py4DSTEM, and real backends

This companion notebook is a **map, not a re-implementation**. The clean twin
(`clean_kinematical` build) does fast **kinematical** diffraction, imaging, EELS, thickness
selection, drift/damage/contamination, feature-finding, and the vendor-backend abstraction.
Everything below is **already demonstrated and working** in the accompanying notebooks — here
we state what each extension gives you, point to where to see it run, and say **exactly where
in the code a real backend plugs in**.

The design that makes all of this clean is the **two-path architecture**:

- **Interactive path (default):** the twin server's kinematical engine. Fast (~seconds),
  correct spot positions, no heavy dependencies. This is what you develop acquisition and
  decision logic against.
- **High-fidelity / offline path:** external engines (abTEM, later Prismatic) that consume the
  *same* atoms the twin exposes, to generate analysis-grade data. Slower, run offline.

Both paths read atoms through one interface every sample implements:

```python
positions_A, Z = sample.get_atoms_in_region(cx_um, cy_um, half_width_um, depth_nm)
```

That single seam is why the twin, abTEM, and py4DSTEM all see a consistent specimen.

## 1. abTEM — dynamical (multislice) diffraction  ·  *working, see the abTEM notebook*

**What it adds over the built-in kinematical DIFF:** physically correct spot *intensities*,
thickness fringes, channelling contrast, Kikuchi lines, and (with frozen phonons) the
thermal-diffuse background — the channels quantitative tools read (thickness, atom-counting,
strain, ptychography).

**Status:** implemented and validated as the module `abtem_diffraction.py`, integrated into the
*with-abTEM* twin as Section 11 (kinematical-vs-dynamical comparison, dynamical diffraction of
the twin's own samples, 4D-STEM export, and a specimen-tilt series). It runs on Colab's NumPy
2.x with no downgrade.

**Where the code lives / where a real backend plugs in:**
- Module: `abtem_diffraction.py` — class `AbtemDiffraction` with `saed`, `cbed`, `scan_4d`,
  `build_crystal`, `build_crystal_tilted`, `atoms_from_twin_sample`, `show`, `save_4d`.
- Twin bridge: `AbtemDiffraction.atoms_from_twin_sample(sample, ...)` calls the sample's
  `get_atoms_in_region` — **this is the seam** between the twin and any multislice engine.
- To swap in a different engine (e.g. **Prismatic/pyprismatic** for GPU-accelerated large 4D
  scans): implement the same three methods (`saed`/`cbed`/`scan_4d`) over your engine, taking
  an ASE `Atoms` in and returning the same arrays out. Nothing else in the twin changes.

→ **See it run:** `..._with_abTEM.ipynb`, Section 11, and the standalone
`STEM_Digital_Twin_abTEM_Diffraction_Module.ipynb`.

## 2. py4DSTEM — 4D-STEM analysis  ·  *working, round-trip shown*

**What it adds:** the standard analysis library for 4D-STEM data — virtual imaging, strain and
orientation mapping, disk detection, ptychography. It is a **consumer** of the 4D datasets the
twin (via abTEM) generates, which is the natural fit for a testbed: develop/validate your
analysis code against known-ground-truth simulated data before beamtime.

**Status:** the round-trip is demonstrated — `scan_4d` → save `.npy` →
`py4DSTEM.DataCube(data=np.load(...))` → virtual bright-field / mean pattern. The clean build
also shows the identical operations in **plain NumPy** so no extra dependency is required.

**Packaging caveat (important):** py4DSTEM pins **`numpy<2`**, so installing it in Colab
downgrades NumPy and requires a **runtime restart**. Keep it isolated: generate + save the 4D
data with abTEM (NumPy 2.x), then in a *separate* step install py4DSTEM and analyze the saved
file. Don't install it alongside the twin/abTEM.

**Where a real backend plugs in:** py4DSTEM reads a saved 4D array — there is no twin-side code
to change. If you later record 4D data on a real microscope, save it in the same
`(scan_y, scan_x, det_y, det_x)` layout (or py4DSTEM's native `.h5`/EMD) and the same analysis
cells work unchanged.

→ **See it run:** the abTEM module notebook (Section 5 / 5b) and the appendix.

## 3. Real-microscope backends (ThermoFisher / Nion / JEOL-PyJEM)  ·  *interface in place*

**What it is:** the twin and every vendor adapter implement one identical instrument-control
interface, so a workflow written against the twin deploys to hardware by swapping the backend
object — no workflow changes.

**Where the code lives / where a real backend plugs in:**
- `microscope_backend.py` — abstract `MicroscopeBackend` plus `TwinBackend`,
  `ThermoFisherBackend`, `NionBackend`, `JEOLBackend`. All share the same 10-method surface:
  `get_stage/set_stage`, `get_beam/set_beam`, `set_mode`, `set_fov_um`,
  `get_magnification/set_magnification`, `acquire_image`, `autofocus`.
- The vendor classes are **honest skeletons**: method bodies contain the real SDK calls
  (AutoScript for ThermoFisher, `nion.instrumentation` for Nion, PyJEM `TEM3` for JEOL) but are
  **unvalidated on hardware**. To deploy: fill in / verify those SDK calls for your instrument
  and instantiate that backend instead of `TwinBackend`. The beam-safety pattern
  (`set_beam(..., disabled=True)`) and units (µm / degrees / pA / kV / magnification) are shared.

**EELS on a real backend:** the twin exposes `set_mode("EELS")` + `acquire_spectrum(...)`. On a
real instrument these map to the spectrometer's acquisition call returning a 1-D spectrum; add
an `acquire_spectrum` to the vendor backend that wraps the SDK's EELS acquisition and returns the
same `{energy_ev, intensity, ...}` dict.

## 4. Where each extension is added — quick reference

| Extension | File to edit | What to add |
|---|---|---|
| Dynamical diffraction (abTEM) | `abtem_diffraction.py` | already implemented; use `atoms_from_twin_sample` seam |
| Alt. multislice engine (Prismatic/GPU) | new module, mirror `AbtemDiffraction` | `saed/cbed/scan_4d` over your engine; same in/out |
| 4D-STEM analysis (py4DSTEM) | *no twin code* | consume saved `.npy` in an isolated (numpy<2) env |
| Real instrument control | `microscope_backend.py` | fill vendor SDK calls in `ThermoFisher/Nion/JEOL` backends |
| Real EELS | `microscope_backend.py` | add `acquire_spectrum` wrapping the spectrometer SDK |
| New sample from real atoms | `samples/atomsk_polycrystal.py` (pattern) | load a file, expose `get_atoms_in_region` |

**The one rule that keeps it all consistent:** anything that needs the specimen's atoms goes
through `get_atoms_in_region`. The interactive twin, abTEM, a future Prismatic backend, and EELS
composition all read from that same method, so a change to a sample is seen everywhere at once.

## 5. Suggested reading order

1. **`..._clean_kinematical.ipynb`** — the working twin: samples, imaging, kinematical
   diffraction, thickness selection, EELS, drift/damage, feature-finding, backends. Start here.
2. **`STEM_Digital_Twin_abTEM_Diffraction_Module.ipynb`** — the dynamical-diffraction module in
   isolation (SAED, CBED, 4D-STEM, tilt series), with the twin bridge.
3. **`..._with_abTEM.ipynb`** — the same twin with abTEM integrated as Section 11.
4. **`STEM_Digital_Twin_Appendix_abTEM_multislice.ipynb`** — the standalone appendix incl. the
   py4DSTEM round-trip and the packaging notes.

Everything above is runnable today on CPU; a GPU only matters for large 4D scans.